In [1]:
from glob import glob
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import constants as C

In [2]:
RESET_MERGE = False

In [3]:
def concat_yearly_data(file_list, usecols):
	data_frames = []
	for file in file_list:
		df = pd.read_csv(file, usecols=usecols, encoding="latin-1", low_memory=False)
		year = file.split('\\')[1].replace('FARS', '').replace('NationalCSV', '')
		df['YEAR'] = int(year)
		data_frames.append(df)
	combined_df = pd.concat(data_frames, ignore_index=True)
	return combined_df


def merge_dataframes(acc_df, dist_df, veh_df, pers_df):
	distract_vehicle = dist_df.merge(
		veh_df, on=['ST_CASE', 'YEAR', 'VEH_NO'], how='left')
	distract_vehicle_accident = distract_vehicle.merge(
		acc_df, on=['ST_CASE', 'YEAR'], how='left')
	full_merged = distract_vehicle_accident.merge(
		pers_df, on=['ST_CASE', 'YEAR', 'VEH_NO'], how='left')
	return full_merged

In [4]:
if RESET_MERGE:
	accident_files = glob(C.ACCIDENT_PATH)
	print(f"Number of accident files found: {len(accident_files)}")

	accident_concatenated = concat_yearly_data(accident_files, C.accident_cols)
	print(f"Concatenated ACCIDENT data shape: {accident_concatenated.shape}")
	print("Columns in concatenated ACCIDENT data:")
	print(accident_concatenated.columns)

	#################################################################################
	distract_files = glob(C.DISTRACT_PATH)
	print(f"Number of distraction files found: {len(distract_files)}")

	distract_concatenated = concat_yearly_data(distract_files, C.distract_cols)
	print(f"Concatenated DISTRACTION data shape: {distract_concatenated.shape}")
	print("Columns in concatenated DISTRACTION data:")
	print(distract_concatenated.columns)

	#################################################################################
	vehicle_files = glob(C.VEHICLE_PATH)
	print(f"Number of vehicle files found: {len(vehicle_files)}")

	vehicle_concatenated = concat_yearly_data(vehicle_files, C.vehicle_cols)
	print(f"Concatenated VEHICLE data shape: {vehicle_concatenated.shape}")
	print("Columns in concatenated VEHICLE data:")
	print(vehicle_concatenated.columns)

	#################################################################################
	person_files = glob(C.PERSON_PATH)
	print(f"Number of person files found: {len(person_files)}")

	person_concatenated = concat_yearly_data(person_files, C.person_cols)
	print(f"Concatenated PERSON data shape: {person_concatenated.shape}")
	print("Columns in concatenated PERSON data:")
	print(person_concatenated.columns)

	#################################################################################
	df = merge_dataframes(accident_concatenated, distract_concatenated,
	                      vehicle_concatenated, person_concatenated)
	print(f"Merged DataFrame shape: {df.shape}")
	df.to_csv(C.MERGED_DATA_PATH, index=False)

else:
	df = pd.read_csv(C.MERGED_DATA_PATH, low_memory=False)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 726930 entries, 0 to 726929
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   ST_CASE   726930 non-null  int64  
 1   VEH_NO    726930 non-null  int64  
 2   MDRDSTRD  726930 non-null  int64  
 3   YEAR      726930 non-null  int64  
 4   FATALS    726930 non-null  int64  
 5   PER_TYP   723888 non-null  float64
 6   INJ_SEV   723888 non-null  float64
dtypes: float64(2), int64(5)
memory usage: 38.8 MB


In [6]:
df.head()

,ST_CASE,VEH_NO,MDRDSTRD,YEAR,FATALS,PER_TYP,INJ_SEV
0,10001,1,0,2010,1,1.0,4.0
1,10001,1,0,2010,1,2.0,3.0
2,10002,1,0,2010,1,1.0,4.0
3,10003,1,0,2010,1,1.0,4.0
4,10003,2,0,2010,1,1.0,0.0


In [ ]:
def apply_distraction_category(row):
	for category, codes in C.distraction_categories.items():
		if row['MDRDSTRD'] in codes:
			return category
	return 'Unknown'

df_fatal_drivers = df.copy()
df_fatal_drivers['DISTRACT_GROUP'] = df_fatal_drivers.apply(apply_distraction_category, axis=1)
df_fatal_drivers.head()

## Preprocessing

In [ ]:
# Convert to minimal dtypes
for col in ['YEAR', 'DRDISTRACT', 'PERTYP']:
    if col in df.columns:
        df[col] = df[col].astype('int16')
df['FATALS'] = df['FATALS'].astype('int8')

In [ ]:
df.head()

In [ ]:
# Heatmap of correlations
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
grouped = df.groupby(['YEAR', 'MDRDSTRD'])['FATALS'].sum().reset_index()

# Replace MDRDSTRD codes with category names
def map_distraction_category(code):
	for category, codes in C.distraction_categories.items():
		if code in codes:
			return category

grouped['MDRDSTRD'] = grouped['MDRDSTRD'].apply(map_distraction_category)
grouped

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=summary.reset_index().melt(id_vars='YEAR', var_name='Distraction Category', value_name='Fatalities'), x='YEAR', y='Fatalities', hue='Distraction Category', marker='o')
plt.title('Distracted Driving Fatalities by Distraction Category (2010-2019)')
plt.xlabel('Year')
plt.ylabel('Number of Fatalities')
plt.legend(title='Distraction Category')
plt.show()